In [ ]:
import jax
import jax.numpy as jnp
import sys  
import numpy as np
import matplotlib.pyplot as plt 
import jax.config as jax_config
import os
path = "../TRAINING"
os.chdir(path)
os.listdir(path)
from model import create_model
jax_config.update("jax_enable_x64", True)

In [ ]:
DIM = 2
TRAIN_ITERATION = 10000
BATCH_SIZE = 100
DT = 1e-2
FEATURE_NET_LAYER = [12,24, 48]
DESCRIBE_NET_LAYER = [128, 128, 128, DIM]

params = jnp.load(f"./model_save_path/jax_ckpt_000010.npz", allow_pickle=True)['params'].item()
_, apply_fn = create_model(FEATURE_NET_LAYER, DESCRIBE_NET_LAYER, DIM)

In [ ]:
def ic_ring(N, R=1.0, thickness=0.05, seed=0):
    rng=np.random.default_rng(seed)
    N = N//2
    angles=rng.uniform(0,2*np.pi,size=N)
    radii=R + rng.normal(0, thickness, size=N)
    x1 = 0.5+np.stack([radii*np.cos(angles), radii*np.sin(angles)],1)
    x2 = -0.5 + np.stack([radii*np.cos(angles), radii*np.sin(angles)],1)
    return np.concatenate([x1, x2], axis=0).astype(np.float64)
x = jnp.array(ic_ring(16000))

In [ ]:
def step(xn):
    f = apply_fn(params, x)
    return xn + DT *f


In [ ]:
x_save = x[None,...]
for i in range(200):
    x = step(x)
    x_save = jnp.concatenate([x_save, x[None,...]], axis=0)


In [ ]:
plt.hist2d(x_save[0,:,0],x_save[0,:,1],bins=100);

In [ ]:
plt.hist2d(x_save[-1,:,0],x_save[-1,:,1],bins=100);